# 20) Model Analysis and Interpretability (aFRR)

Dieses Notebook liefert eine tiefgehende Evaluierung eines trainierten
XGBoost-Modells fuer die Zielvariable
`target_afrr_activation_price_vwap_pos_h1`.

Inhalte:
1. Setup & Daten/Modell-Laden
2. Performance-Audit (MAE, RMSE, Scatter, letzte 14 Tage)
3. SHAP-Interpretierbarkeit (Summary + Dependence)
4. Bivariate Residual-Analyse
5. Business Case: Theoretischer Profit (Spread-Richtungslogik)


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

from energy_trading.visualization.style import apply_geo_style, THESIS_PALETTE

warnings.filterwarnings('ignore')
apply_geo_style()


def resolve_repo_root() -> Path:
    root = Path.cwd().resolve()
    if (root / 'src').exists():
        return root
    for p in root.parents:
        if (p / 'src').exists():
            return p
    raise RuntimeError("Could not resolve REPO_ROOT (directory containing 'src').")


REPO_ROOT = resolve_repo_root()
FEATURE_PATH = REPO_ROOT / 'data/features/all_data_features.parquet'
CONFIG_PATH = REPO_ROOT / 'data/model_input/feature_config.json'
REPORT_DIR = REPO_ROOT / 'data/reports/model_analysis'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Standard: vorhandenes Joblib-Modell. Optional auf model.json umstellbar.
MODEL_PATH = REPO_ROOT / 'models/checkpoints/xgboost_da_v1.joblib'
TARGET_COL = 'target_afrr_activation_price_vwap_pos_h1'

print('REPO_ROOT:', REPO_ROOT)
print('FEATURE_PATH:', FEATURE_PATH)
print('CONFIG_PATH:', CONFIG_PATH)
print('MODEL_PATH:', MODEL_PATH)


In [ ]:
assert FEATURE_PATH.exists(), f'Missing feature artifact: {FEATURE_PATH}'
assert CONFIG_PATH.exists(), f'Missing feature config: {CONFIG_PATH}'
assert MODEL_PATH.exists(), f'Missing model file: {MODEL_PATH}'

cfg = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
afrr_cfg = cfg['bundles']['afrr']
feature_cols = afrr_cfg['features']

if TARGET_COL not in afrr_cfg['targets']:
    print(f"[WARN] Target {TARGET_COL} not listed in afrr targets: {afrr_cfg['targets']}")

df = pd.read_parquet(FEATURE_PATH)
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True, errors='coerce')
df = df.sort_values('timestamp_utc').reset_index(drop=True)

if TARGET_COL not in df.columns:
    raise KeyError(f'Missing target column in artifact: {TARGET_COL}')

# Rekonstruiere Split-Grenzen aus feature_config (inkl. Purge-Gap).
train_end = pd.Timestamp(cfg['splits']['train_end_exclusive'])
val_end = pd.Timestamp(cfg['splits']['val_end_exclusive'])
gap = int(cfg['splits'].get('purge_gap_rows', 72))

df_nonan_target = df[df[TARGET_COL].notna()].copy().reset_index(drop=True)
idx_train_end = int((df_nonan_target['timestamp_utc'] < train_end).sum())
idx_val_end = int((df_nonan_target['timestamp_utc'] < val_end).sum())

val_start = min(idx_train_end + gap, len(df_nonan_target))
test_start = min(idx_val_end + gap, len(df_nonan_target))

df_test = df_nonan_target.iloc[test_start:].copy()
if df_test.empty:
    raise ValueError('Test split is empty after applying split bounds + purge gap.')

missing_feats = [c for c in feature_cols if c not in df_test.columns]
if missing_feats:
    raise KeyError(f'Missing expected feature columns in artifact (sample): {missing_feats[:10]}')

X_test = df_test[feature_cols].copy()
y_test = pd.to_numeric(df_test[TARGET_COL], errors='coerce')

# timestamp as metadata/index only
X_test.index = pd.to_datetime(df_test['timestamp_utc'], utc=True)
y_test.index = X_test.index

print('Rows total:', len(df), '| rows with target:', len(df_nonan_target), '| test rows:', len(X_test))
print('Test span:', X_test.index.min(), '->', X_test.index.max())


In [ ]:
# Modell laden (joblib oder model.json)
model = None
if MODEL_PATH.suffix.lower() == '.json':
    import xgboost as xgb
    model = xgb.Booster()
    model.load_model(str(MODEL_PATH))
    # Fuer Booster braucht man DMatrix fuer Predict.
    model_type = 'booster_json'
else:
    model = joblib.load(MODEL_PATH)
    model_type = 'joblib_estimator'

print('Model type:', model_type)
print(type(model))


In [ ]:
# Vorhersage + Performance
if model_type == 'booster_json':
    import xgboost as xgb
    dtest = xgb.DMatrix(X_test)
    y_pred = pd.Series(model.predict(dtest), index=X_test.index)
else:
    # XGBoost sklearn wrapper erwartet DataFrame mit gleichen Feature-Namen.
    y_pred = pd.Series(model.predict(X_test), index=X_test.index)

mae = float(mean_absolute_error(y_test, y_pred))
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))

metrics = pd.DataFrame([
    {'metric': 'MAE', 'value': mae},
    {'metric': 'RMSE', 'value': rmse},
    {'metric': 'n_test_rows', 'value': float(len(X_test))},
])
metrics


In [ ]:
# Scatter: Actual vs Predicted
plt.figure(figsize=(7, 7))
plot_df = pd.DataFrame({'actual': y_test, 'pred': y_pred}).dropna()

sns.scatterplot(
    data=plot_df,
    x='actual',
    y='pred',
    alpha=0.5,
    s=24,
    color=THESIS_PALETTE['primary'],
)

lo = float(min(plot_df['actual'].min(), plot_df['pred'].min()))
hi = float(max(plot_df['actual'].max(), plot_df['pred'].max()))
plt.plot([lo, hi], [lo, hi], color=THESIS_PALETTE['tertiary'], linewidth=2, label='Ideal: y=x')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted (aFRR Target)')
plt.legend()
plt.tight_layout()
out_scatter = REPORT_DIR / 'actual_vs_predicted_scatter.png'
plt.savefig(out_scatter, dpi=160)
plt.show()
out_scatter


In [ ]:
# Zeitreihenplot: letzte 14 Tage im Test
plot_ts = pd.DataFrame({'actual': y_test, 'pred': y_pred}).dropna().copy()
if not plot_ts.empty:
    end_ts = plot_ts.index.max()
    start_ts = end_ts - pd.Timedelta(days=14)
    w = plot_ts.loc[plot_ts.index >= start_ts]

    plt.figure(figsize=(14, 5))
    plt.plot(w.index, w['actual'], color=THESIS_PALETTE['primary'], label='Actual', linewidth=2)
    plt.plot(w.index, w['pred'], color=THESIS_PALETTE['secondary'], label='Predicted', linewidth=2)
    plt.title('Last 14 Days: Actual vs Predicted')
    plt.xlabel('timestamp_utc')
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.tight_layout()
    out_ts = REPORT_DIR / 'last14d_timeseries_actual_vs_pred.png'
    plt.savefig(out_ts, dpi=160)
    plt.show()
    out_ts
else:
    print('No data available for 14-day plot.')


In [ ]:
# SHAP Summary + Top-2 Dependence
# Hinweis: Bei sehr kleinem Testset sind SHAP-Plots statistisch begrenzt, aber technisch aussagefaehig.
try:
    import shap

    if len(X_test) > 5000:
        X_shap = X_test.sample(5000, random_state=42)
    else:
        X_shap = X_test.copy()

    if model_type == 'booster_json':
        # TreeExplainer kann auch Booster direkt.
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.TreeExplainer(model)

    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_values = np.asarray(shap_values)

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_shap, show=False, max_display=20)
    plt.tight_layout()
    out_shap = REPORT_DIR / 'shap_summary_top20.png'
    plt.savefig(out_shap, dpi=160)
    plt.show()

    shap_mean_abs = np.abs(shap_values).mean(axis=0)
    top_idx = np.argsort(shap_mean_abs)[::-1][:2]
    top_feats = [X_shap.columns[i] for i in top_idx]
    print('Top SHAP features:', top_feats)

    for feat in top_feats:
        plt.figure(figsize=(8, 5))
        shap.dependence_plot(feat, shap_values, X_shap, interaction_index=None, show=False)
        plt.tight_layout()
        out_dep = REPORT_DIR / f'shap_dependence_{feat}.png'
        plt.savefig(out_dep, dpi=160)
        plt.show()

except Exception as e:
    print('[WARN] SHAP analysis skipped:', e)


In [ ]:
# Bivariate Residual-Analyse
residual = y_test - y_pred
res_df = pd.DataFrame({'residual': residual}, index=X_test.index)

# Kandidaten gemaess Anfrage / Verfuegbarkeit
cand_grid = [c for c in ['grid_stress_index_lag_2h', 'grid_stress_index'] if c in X_test.columns]
cand_solar = [c for c in ['solar_forecast_id_entsoe', 'solar_forecast_da_entsoe'] if c in X_test.columns]

if not cand_grid:
    print('[WARN] No grid stress column found in X_test.')
if not cand_solar:
    print('[WARN] No solar forecast column found in X_test.')

plot_pairs = []
if cand_grid:
    plot_pairs.append(cand_grid[0])
if cand_solar:
    plot_pairs.append(cand_solar[0])

if plot_pairs:
    fig, axes = plt.subplots(1, len(plot_pairs), figsize=(7 * len(plot_pairs), 5))
    if len(plot_pairs) == 1:
        axes = [axes]

    for ax, feat in zip(axes, plot_pairs):
        tmp = pd.DataFrame({'x': pd.to_numeric(X_test[feat], errors='coerce'), 'residual': residual}).dropna()
        sns.regplot(
            data=tmp,
            x='x',
            y='residual',
            scatter_kws={'alpha': 0.35, 's': 20, 'color': THESIS_PALETTE['primary']},
            line_kws={'color': THESIS_PALETTE['tertiary'], 'linewidth': 2},
            ax=ax,
        )
        ax.axhline(0, color=THESIS_PALETTE['neutral_dark'], linewidth=1)
        ax.set_title(f'Residuals vs {feat}')
        ax.set_xlabel(feat)

    plt.tight_layout()
    out_res = REPORT_DIR / 'residual_bivariate_checks.png'
    plt.savefig(out_res, dpi=160)
    plt.show()
    out_res


In [ ]:
# Business Case: Theoretischer Profit (Spread-Richtung)
# Spread_t = aFRR_price_t - DA_price_t
# Trading-Entscheidung basiert auf predicted spread sign; Auszahlung nach actual spread sign.

if 'da_price_pit' not in X_test.columns:
    raise KeyError("Business metric requires da_price_pit in X_test.")

da = pd.to_numeric(X_test['da_price_pit'], errors='coerce')
spread_pred = y_pred - da
spread_true = y_test - da

m = spread_pred.notna() & spread_true.notna()
spread_pred = spread_pred[m]
spread_true = spread_true[m]

# Richtungsentscheidung (+1: discharge/verkaufen, -1: charge/kaufen)
signal = np.sign(spread_pred)

# Theoretischer PnL-Proxy: richtiger Richtungstreffer verdient |true spread|,
# falscher verliert |true spread|.
pnl_hour = np.where(signal * np.sign(spread_true) >= 0, np.abs(spread_true), -np.abs(spread_true))

pnl_df = pd.DataFrame({
    'spread_pred': spread_pred,
    'spread_true': spread_true,
    'signal': signal,
    'pnl_hour_eur_per_mwh_proxy': pnl_hour,
}, index=spread_true.index)

directional_accuracy = float((np.sign(spread_pred) == np.sign(spread_true)).mean())
total_pnl_proxy = float(np.nansum(pnl_hour))
mean_pnl_proxy = float(np.nanmean(pnl_hour))

summary = pd.DataFrame([
    {'metric': 'directional_accuracy', 'value': directional_accuracy},
    {'metric': 'total_pnl_proxy', 'value': total_pnl_proxy},
    {'metric': 'mean_pnl_proxy', 'value': mean_pnl_proxy},
    {'metric': 'n_hours', 'value': float(len(pnl_df))},
])

out_pnl = REPORT_DIR / 'business_case_theoretical_pnl_proxy.csv'
summary.to_csv(out_pnl, index=False)

summary


## Hinweis zur Interpretation

- Dieses Notebook bewertet ein bestehendes Modell auf dem aFRR-Target.
- Wenn ein DA-Modell geladen wird (statt eines explizit auf aFRR trainierten
  Modells), sind die Metriken als technischer Check zu verstehen, nicht als
  finaler Modellvergleich fuer die Thesis.
- Fuer finale Thesis-Auswertung sollte ein dediziertes aFRR-Modell genutzt
  werden (gleiches Feature-Set, aFRR-Target als Trainingsziel).
